# Trace an OpenAI Agents SDK support-triage system

You will build a support desk out of three agents. A **Triage** agent reads the incoming message
and answers nothing — it decides who should, and **hands off** to a Billing agent or a Tech Support
agent, each with its own tool. Then you will run a two-turn conversation through it and watch the
whole thing arrive in AcruxCore.

There is no AcruxCore code in the agents. The interesting part of this notebook is *how* that is
possible, and it is not the same mechanism as the CrewAI tutorial's. The
[OpenAI Agents SDK](https://github.com/openai/openai-agents-python) has a **tracing pipeline of its
own**, with a plug-in point for trace processors. Nothing patches it from outside. Instead a
processor is plugged in, and that has one consequence worth the whole of Step 1: switch the SDK's
tracing off and you switch AcruxCore off with it.

| Piece | What it does | Who runs it |
|---|---|---|
| `register()` | builds an OTel pipeline and plugs a processor into the Agents SDK | **your code**, once at startup |
| `openinference-instrumentation-openai-agents` | that processor: turns SDK trace events into spans | a library |
| Triage / Billing / Tech Support | read, route, answer | **the Agents SDK**, knowing nothing about us |
| `check_subscription`, `lookup_order` | two deliberately fake lookups | **your code** |
| `using_session(...)` | groups both turns as one conversation | **your code** |

The two tools return canned data on purpose. This notebook is about the **shape** of the trace — a
handoff between two agents — so a real API behind the tool would only add noise and a third key.

Every cell runs against a real account and the real OpenAI API. The tracing is real; only the two
lookups are mocked.

**Two ways to do every step.** The one step that changes something on the platform has two
headings: **In the dashboard**, with the values to set, and **The same thing in code**. They are not
two different features — the dashboard and the SDK call the same API. Pick either.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | changes something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Trace an OpenAI Agents SDK Support-Triage System](https://docs.acruxcore.com/docs/tutorials/trace-an-openai-agents-sdk-triage-system)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**. Copy it the moment it appears — that is
the only time the full value is shown.

**2. An OpenAI API key.** The agents call `gpt-4o-mini` directly through the Agents SDK, so this key
goes to OpenAI and never to us. This is the one notebook in the set that really is tied to one
provider: the Agents SDK is OpenAI's own client, and pointing it elsewhere means constructing a
different model object, not changing a string.

**3. Nothing else.** No Tavily key, no gateway model, no prompts. Both tools are mocked, and this
whole notebook costs well under a cent.

**4. Three packages.** The framework, our OTel helper, and the processor that connects them.

In [ ]:
%pip install -q --upgrade openai-agents "acruxcore[otel]" \
    openinference-instrumentation-openai-agents

**Setup.** Two keys and one session name.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your shell
before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("OPENAI_API_KEY", "sk-...")

MODEL = "gpt-4o-mini"
SESSION_ID = "support-triage-notebook"

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Three things, in the order they fail. Read the versions even when they pass: every span
name in this notebook comes from the instrumentor, and those names change between releases.

In [2]:
import importlib.metadata as metadata

import requests

from acruxcore import AcruxCore

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the AcruxCore key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# 2. Which versions? Span names and shapes come from these two.
for package in ("openai-agents", "openinference-instrumentation-openai-agents",
                "acruxcore", "opentelemetry-sdk"):
    print(f"  {package:<48} {metadata.version(package)}")

# 3. Does the OpenAI key work? The agents call OpenAI directly, so we never see this failing.
probe = requests.post("https://api.openai.com/v1/chat/completions", timeout=60,
                      headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
                      json={"model": MODEL, "max_tokens": 1,
                            "messages": [{"role": "user", "content": "hi"}]})
print("openai key:", "ok" if probe.ok else f"FAILED {probe.status_code}")

acruxcore key: ok
  openai-agents                                    0.22.0
  openinference-instrumentation-openai-agents      2.0.0
  acruxcore                                        0.10.0
  opentelemetry-sdk                                1.42.1
openai key: ok


---

## Step 1 — How a framework's telemetry gets redirected, and what a handoff looks like

### The general problem

To see inside somebody else's framework you have to get code in there somehow, and there are two
ways in.

**Patch it from outside.** Wrap the framework's methods at import time and record calls as they pass
through. This works on any library, invited or not, and it is what the CrewAI instrumentor does.

**Use a plug-in point the framework already provides.** Some frameworks emit their own telemetry and
let you register a receiver for it. Then you are not fighting the library; you are a consumer of a
stream it was already producing.

The second is cleaner and more stable across releases. It has one property the first does not: you
are downstream of the framework's own switch.

### Where our case sits

The Agents SDK has its own tracing pipeline, which by default reports to
`platform.openai.com`. `openinference-instrumentation-openai-agents` **replaces that default trace
processor with its own**, which converts each SDK trace event into an OTel span.
`acruxcore.otel.register()` builds the pipeline those spans flow into — a `TracerProvider`, a
`BatchSpanProcessor` and an `OTLPSpanExporter` aimed at `$ACRUXCORE_BASE_URL/traces/otlp`.

So the path is: **Agents SDK tracing → OpenInference processor → OTel exporter → us.**

### The direct answer: never disable the SDK's tracing

It is a natural first move. You are sending traces somewhere else now, so why leave OpenAI's own
tracing on? Because `set_tracing_disabled(True)` (Python) or `tracingDisabled: true` (Node) does not
switch off *a destination* — it switches off the **pipeline**, which is the first link in the chain
above. The OpenInference processor then receives nothing, so we receive nothing.

There is no separate disable step to perform. `instrument=["openai_agents"]` already replaced the
processor that talked to OpenAI. Step 8 turns tracing off for real and shows the traces stop
arriving, with no error anywhere.

### And a handoff is a tool span

A handoff is the Agents SDK's signature feature, and AcruxCore has no `handoff` span kind. What
arrives is a **`tool` span named `handoff to Tech Support`**, with the source agent as its input and
the target agent as its output:

```
tool  handoff to Tech Support   input: "Triage"   output: "Tech Support"
```

That is not an approximation we invented. OpenInference represents a handoff as the calling agent
invoking a tool named for the target, and we map span kinds from the OpenInference attributes we
receive. Inventing a kind for one framework's concept would make the same trace look different
depending on which library reported it.

### Two kinds of "continuing", and they are unrelated

This trips people up, so it is worth separating before Step 6.

`turn1.to_input_list() + [...]` continues the **conversation**: it feeds the whole message history
back into the next run so the agents remember what was said. That is Agents SDK behaviour and
nothing to do with tracing.

`using_session("…")` continues the **observability session**: it stamps a `session.id` attribute on
every span so we group the runs. That is tracing and nothing to do with the conversation.

You can have either without the other. A conversation without a session is two traces nobody can
relate; a session without the conversation history is two turns that forgot each other.

### The recommendation

Call `register()` once, before you build your agents, and then leave the SDK's tracing alone. If you
want the agents' work grouped per user or per ticket, wrap each request in `using_session` with an
id you already have.

---

## Step 2 — Turn on payload capture

Without this, the handoff span still arrives — you can see that Triage handed off, and when. You
cannot see *to whom*, because the target agent's name is in the span's output payload. The same goes
for the tool call's arguments.

The setting is per team and applies to every trace, so it is a deliberate choice about storing model
and tool payloads rather than a per-run flag.

### In the dashboard

**Observability → Settings.**

| Field | What to set |
|---|---|
| **Capture payloads** | on |

### The same thing in code

**Setup.** Read first, write only if it differs — an unnecessary write is still an audit entry.

In [3]:
settings = await hub.traces.get_settings()
print("capture_payloads is:", settings.capture_payloads)

if not settings.capture_payloads:
    settings = await hub.traces.update_settings(capture_payloads=True)
    print("turned it on:", settings.capture_payloads)
else:
    print("already on, nothing to change")

capture_payloads is: True
already on, nothing to change


---

## Step 3 — Register the pipeline

**Your app.** Three lines, before the agents are built.

`instrument=["openai_agents"]` is the line that swaps the SDK's trace processor. Keep the provider
it returns: you flush with it.

Run this cell **once**. OpenTelemetry allows one global provider per process, so a second call
cannot replace it and hands back a provider that is not the one exporting your spans. To change the
configuration in a notebook, restart the kernel.

In [4]:
from acruxcore.otel import register

provider = register(
    service_name="support-triage-agents-sdk",
    instrument=["openai_agents"],
)
print("provider:", type(provider).__name__)
print("exporting to: $ACRUXCORE_BASE_URL/traces/otlp")

provider: TracerProvider
exporting to: $ACRUXCORE_BASE_URL/traces/otlp


---

## Step 4 — Write the two tools

There is nothing to do in the dashboard for this step, and nothing arrives in the tool catalog
either. These tools belong to the Agents SDK, not to us — we only see them being called.

**Your app.** `@function_tool` derives each tool's name, description and argument schema from the
function's signature and docstring, exactly like our own decorator does. The data is canned so the
notebook has no third dependency.

In [5]:
from agents import Agent, Runner, function_tool

MOCK_SUBSCRIPTIONS = {
    "alex@example.com": {"tier": "Pro", "renewed_on": "2026-08-01", "last_charge_usd": 49.00},
}
MOCK_ORDERS = {
    "A1234": {"status": "delivered", "app_version": "3.4.1", "known_crash_bug": True},
}


@function_tool
def check_subscription(customer_email: str) -> str:
    """Look up a customer's subscription tier and most recent charge.

    Args:
        customer_email: The customer's account email address.
    """
    record = MOCK_SUBSCRIPTIONS.get(customer_email)
    if not record:
        return f"No subscription found for {customer_email}."
    return (f"{customer_email} is on the {record['tier']} plan, renewed "
            f"{record['renewed_on']}, last charge ${record['last_charge_usd']:.2f}.")


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up an order's delivery status and the app version tied to it.

    Args:
        order_id: The order identifier, e.g. "A1234".
    """
    record = MOCK_ORDERS.get(order_id)
    if not record:
        return f"No order found with id {order_id}."
    note = (" This app version has a known crash bug, already fixed in the latest release."
            if record["known_crash_bug"] else "")
    return (f"Order {order_id} was {record['status']} on app version "
            f"{record['app_version']}.{note}")


print("tools:", check_subscription.name, "|", lookup_order.name)

tools: check_subscription | lookup_order


---

## Step 5 — Build the three agents

**Your app.** Two specialists, then a router that owns neither tool.

The field that makes routing work is `handoff_description`. `instructions` tell an agent how to do
its job; `handoff_description` tells *another* agent when to pick it. Triage reads those two
sentences and nothing else about its colleagues, so they are where routing goes right or wrong.

Note what Triage does **not** have: tools. It is not supposed to answer, and giving it a tool is the
quickest way to make it try.

In [6]:
billing_agent = Agent(
    name="Billing",
    handoff_description="Handles subscription, billing, and charge questions.",
    instructions=("You help with billing questions. Use check_subscription to look up the "
                  "customer's plan and charges before answering. Be concise."),
    tools=[check_subscription],
    model=MODEL,
)

tech_support_agent = Agent(
    name="Tech Support",
    handoff_description="Handles app crashes, bugs, and order/delivery status.",
    instructions=("You help with technical issues and order status. Use lookup_order to "
                  "check the order before answering. Be concise."),
    tools=[lookup_order],
    model=MODEL,
)

triage_agent = Agent(
    name="Triage",
    instructions=("Route the customer to Billing for subscription/charge questions, or to "
                  "Tech Support for app/order problems. Do not answer directly yourself."),
    handoffs=[billing_agent, tech_support_agent],       # no tools of its own
    model=MODEL,
)

print("triage hands off to:", [handoff.name for handoff in triage_agent.handoffs])

triage hands off to: ['Billing', 'Tech Support']


---

## Step 6 — Run the conversation, two turns

**Your app.** Both turns inside one `using_session` block, and turn 2 built from turn 1's message
history — the two kinds of continuing from Step 1, side by side in eight lines.

Nothing tells Triage where either message should go. Watch which specialist answers each one.

In [7]:
from openinference.instrumentation import using_session

with using_session(SESSION_ID):
    turn_1 = await Runner.run(
        triage_agent,
        "I was charged twice this month, can you check my subscription? "
        "My email is alex@example.com",
    )
    print("--- turn 1 ---")
    print(turn_1.final_output)

    # to_input_list() is the CONVERSATION carrying over: the whole history, replayed.
    turn_2_input = turn_1.to_input_list() + [
        {"role": "user",
         "content": "Also my app keeps crashing on order #A1234, can you check that "
                    "order's status?"},
    ]
    turn_2 = await Runner.run(triage_agent, turn_2_input)
    print("\n--- turn 2 ---")
    print(turn_2.final_output)

--- turn 1 ---
Your subscription is on the Pro plan, and it renewed on August 1, 2026, with a last charge of $49.00. If you see a duplicate charge, it may be a processing error. Please check your bank statement or contact support for further assistance.

--- turn 2 ---
Order #A1234 was delivered on app version 3.4.1, which has a known crash bug. This issue has been fixed in the latest release. Please update your app to resolve the crashing issue.


A billing answer, then a technical one, from the same entry point and with no routing logic in the
code. The second turn also knows it is a continuation — it never asks who you are again.

Which specialist replied is the only visible evidence of the handoff so far. The next step makes it
explicit.

---

## Step 7 — Read both turns back

**Check.** Flush first: spans sit in a batch queue in this process, and Step 8 shows what a read
before the flush returns.

Then look the run up by **session**. An OTLP trace takes its root span's name, which for the Agents
SDK is `Agent workflow` — the same for every run, because it describes the kind of run and not this
one. The session id you chose is what identifies the run you just did.

![The session showing both turns, each its own trace, with real token counts and cost](https://docs.acruxcore.com/img/tutorials/trace-an-openai-agents-sdk-triage-system/02-session.png)

In [8]:
provider.force_flush()          # export whatever is still queued

found = await hub.traces.list(session_id=SESSION_ID, limit=20)

# The list comes back newest first. Sort it, so "turn 1" means turn 1 even after a re-run.
runs = sorted(found.data, key=lambda trace: trace.started_at)

print(f"traces in session {SESSION_ID!r}: {found.total}")
for turn, summary in enumerate(runs, start=1):
    print(f"  turn {turn}: spans={summary.span_count}  tokens={summary.total_tokens}  "
          f"cost=${summary.total_cost_usd}  name={summary.name!r}")

traces in session 'support-triage-notebook': 2
  turn 1: spans=12  tokens=539  cost=$0.0001209  name='Agent workflow'
  turn 2: spans=12  tokens=1051  cost=$0.00019275  name='Agent workflow'


Real token counts and real dollar cost, on model calls the Agents SDK made directly to OpenAI.

**Check.** Now the tree for turn 2 — the turn that routes to Tech Support, and the one worth reading
closely.

![The trace tree: a Triage agent span containing an LLM span and a handoff tool span, then a sibling Tech Support agent span with its own LLM and tool spans](https://docs.acruxcore.com/img/tutorials/trace-an-openai-agents-sdk-triage-system/03-trace-tree.png)

In [9]:
def walk(spans, depth=0):
    """Print the span tree. Indentation is the parent/child relationship."""
    for span in spans:
        tokens = f"  {span.total_tokens}tok" if span.total_tokens else ""
        cost = f"  ${span.cost_usd}" if span.cost_usd else ""
        print("  " * (depth + 1) + f"{span.kind:<7} {(span.name or '')[:34]:<34}{tokens}{cost}")
        walk(span.children, depth + 1)


second = await hub.traces.get(runs[-1].id)       # runs[] is sorted oldest first
walk(second.spans)

  agent   Agent workflow                    
    chain   Agent workflow                    
      agent   Triage                            
        chain   turn                              
          llm     response                            309tok  $5.265e-05
          tool    handoff to Tech Support           
      agent   Tech Support                      
        chain   turn                              
          llm     response                            328tok  $5.73e-05
          tool    lookup_order                      
        chain   turn                              
          llm     response                            414tok  $8.28e-05


Read the indentation. `Triage` makes one model call and then a `tool` span — that is the handoff.
`Tech Support` is its **sibling**, not its child, because the conversation was transferred rather
than nested: the first agent is finished when the second starts.

Every `chain turn` in there is one pass of the SDK's own loop. Two under `Tech Support` means it
called the model, ran a tool, and called the model again with the result.

**Check.** The two `tool` spans, side by side. They have the same span kind and mean completely
different things, and the payloads are what tell them apart.

In [10]:
def every_span(spans):
    for span in spans:
        yield span
        yield from every_span(span.children)


for span in every_span(second.spans):
    if span.kind != "tool":
        continue
    payload = span.payload or {}
    print(f"{span.name}   ({span.latency_ms} ms)")
    print(f"  input:  {json.dumps(payload.get('input'))[:180]}")
    print(f"  output: {json.dumps(payload.get('output'))[:180]}")
    print()

handoff to Tech Support   (0 ms)
  input:  "Triage"
  output: "Tech Support"

lookup_order   (0 ms)
  input:  "{\"order_id\":\"A1234\"}"
  output: "Order A1234 was delivered on app version 3.4.1. This app version has a known crash bug, already fixed in the latest release."



The handoff's input is the agent that gave up the conversation and its output is the agent that took
it. The real tool call's input is the arguments the model chose — `A1234`, which it picked out of the
customer's sentence — and its output is what the function returned.

Both are `tool` spans. That is the mapping from Step 1, and this is where you feel it: filter for
`kind == "tool"` in your own code and handoffs will come back alongside real tool calls.

![The expanded lookup_order span showing the order id as input and the delivery status as output](https://docs.acruxcore.com/img/tutorials/trace-an-openai-agents-sdk-triage-system/04-tool-span.png)

---

## Step 8 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code. Each runs one tiny agent,
so they stay cheap.

### Mistake 1 — disabling the Agents SDK's own tracing

**Broken on purpose.** The one that costs the most time, because the reasoning behind it is sound.
You are exporting to AcruxCore now, so you turn off the SDK's own tracing — and the traces stop,
with no error, no warning, and a pipeline that still looks configured.

Step 1 has the reason: that switch is upstream of the OpenInference processor, not downstream.

In [11]:
from agents import set_tracing_disabled

tiny_agent = Agent(name="Echo", instructions="Reply with one word.", model=MODEL)

set_tracing_disabled(True)               # broken on purpose
with using_session("triage-notebook-tracing-disabled"):
    await Runner.run(tiny_agent, "Say off.")
provider.force_flush()

blocked = await hub.traces.list(session_id="triage-notebook-tracing-disabled", limit=5)
print(f"traces with SDK tracing disabled: {blocked.total}")

set_tracing_disabled(False)              # put it back
with using_session("triage-notebook-tracing-restored"):
    await Runner.run(tiny_agent, "Say on.")
provider.force_flush()

restored = await hub.traces.list(session_id="triage-notebook-tracing-restored", limit=5)
print(f"traces after re-enabling:          {restored.total}")

traces with SDK tracing disabled: 0
traces after re-enabling:          1


Zero, then one. The agent answered correctly both times.

### Mistake 2 — reading before flushing

**Broken on purpose.** `BatchSpanProcessor` batches on purpose — that is what makes OTel cheap
enough to leave on everywhere. The price is that a read straight after a run can find nothing at
all, and a process that exits without flushing loses whatever was still queued.

If you have run this notebook before, both counts will be higher than on a first run. What matters
is that the second is larger.

In [12]:
with using_session("triage-notebook-flush-timing"):
    await Runner.run(tiny_agent, "Say queued.")

before = await hub.traces.list(session_id="triage-notebook-flush-timing", limit=5)
print(f"before flush: {before.total} trace(s)")

provider.force_flush()

after = await hub.traces.list(session_id="triage-notebook-flush-timing", limit=5)
print(f"after flush:  {after.total} trace(s)")

before flush: 0 trace(s)
after flush:  1 trace(s)


### Mistake 3 — forgetting `using_session`

**Broken on purpose.** The run is still traced perfectly. It simply has no `session.id`, so nothing
relates it to the turn before it and the session view will never list it.

It still gets a name — `Agent workflow`, like every other run — so with no session there is
nothing that separates it from the rest, and the timestamp is all you have left.

In [13]:
await Runner.run(tiny_agent, "Say orphan.")     # broken on purpose: no using_session(...)
provider.force_flush()

recent = await hub.traces.list(limit=3)
for summary in recent.data:
    print(f"session={str(summary.session_id):<38} name={summary.name!r}")
print("\nA trace with session=None is invisible to Observability -> Sessions.")

session=None                                   name='Agent workflow'
session=triage-notebook-flush-timing           name='Agent workflow'
session=triage-notebook-tracing-restored       name='Agent workflow'

A trace with session=None is invisible to Observability -> Sessions.


### Mistake 4 — a router with nothing to route to

**Broken on purpose.** Not a wiring mistake — an agent-design one, and it shows what the handoff
span is evidence *of*.

Give Triage the same instructions but an empty `handoffs` list. It is told not to answer directly
and has no way to pass the message on, so it does the only thing left. The trace tells you
immediately: one agent, and no handoff span anywhere.

In [14]:
stranded_triage = Agent(
    name="Triage",
    instructions=("Route the customer to Billing for subscription/charge questions, or to "
                  "Tech Support for app/order problems. Do not answer directly yourself."),
    handoffs=[],                          # broken on purpose: nowhere to go
    model=MODEL,
)

with using_session("triage-notebook-no-handoffs"):
    stranded = await Runner.run(stranded_triage, "My app keeps crashing on order #A1234.")
provider.force_flush()

print("it answered anyway:", stranded.final_output[:160], "\n")

stuck = await hub.traces.list(session_id="triage-notebook-no-handoffs", limit=5)
detail = await hub.traces.get(stuck.data[0].id)
print("agents in the trace:", [s.name for s in every_span(detail.spans) if s.kind == "agent"])
print("tool spans:         ", [s.name for s in every_span(detail.spans) if s.kind == "tool"])

it answered anyway: I recommend reaching out to Tech Support for assistance with your app issue related to order #A1234. They will be able to help you resolve it. 

agents in the trace: ['Agent workflow', 'Triage']
tool spans:          []


---

## Step 9 — Flush before you finish

**Your app.** The last thing your entry point does. In a script it goes at the end of `main()`.

`hub` is only used by this notebook's **Check** cells, but it holds an HTTP pool, so close it too.

In [15]:
provider.force_flush()
await hub.gateway.aclose()
print("flushed")

flushed


---

## What you built

A three-agent support desk where the routing is a model's decision, running a two-turn conversation,
fully traced — agents, the handoff between them, tool calls, model calls, tokens and cost — with the
agents knowing nothing about AcruxCore.

### What of this actually ships

Three lines above your agents, and one at the end:

```python
from acruxcore.otel import register

provider = register(
    service_name="support-triage-agents-sdk",
    instrument=["openai_agents"],          # swaps the SDK's own trace processor
)

from agents import Agent, Runner, function_tool
from openinference.instrumentation import using_session

# ... your agents, exactly as you already wrote them ...

async def main() -> None:
    with using_session("support-triage-demo-session"):
        turn_1 = await Runner.run(triage_agent, "I was charged twice this month...")

        turn_2_input = turn_1.to_input_list() + [{"role": "user", "content": "Also..."}]
        turn_2 = await Runner.run(triage_agent, turn_2_input)

    provider.force_flush()                 # do not skip this
```

Everything else in this notebook was scaffolding:

- the preflight and every **Check** cell read state back to prove a step worked. None of it belongs
  in a request path.
- Step 2 is a one-time team setting, and the dashboard does the same job.
- Step 8 is all deliberately broken, and it leaves four extra traces behind.

### The four rules worth remembering

1. **Never disable the Agents SDK's own tracing.** It is the transport, not a competitor.
2. **`register()` once per process**, before the agents are built.
3. **Flush before you exit**, or lose the tail of every run.
4. **A handoff is a `tool` span**, so anything filtering on `kind == "tool"` sees both.

### What this notebook left in your team

- payload capture on, if it was not already
- six traces: two real turns grouped in one session, plus four from Step 8
- nothing else — no prompts, no tools, no models, because none of this went through us

### Where to go next

- [Trace a CrewAI trip-planning crew](https://docs.acruxcore.com/docs/tutorials/trace-a-crewai-trip-planner)
  — the same OTLP endpoint for a framework that is patched from outside instead of plugged into, with
  a real web-search tool.
- [Send OTel traces to AcruxCore with the SDK helper](https://docs.acruxcore.com/docs/guides/send-otel-traces-with-the-sdk-helper)
  — `register()` on its own: a bare pipeline, one framework, or several at once.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — what else a session gives you once your runs are grouped.